# Nuts & Bolts - YOLO11n @320 (2 classes: bolt, nut)
Dataset: Bolts and Nuts v8 by Kim (CC BY 4.0), https://universe.roboflow.com/kim-sxidz/bolts-and-nuts-vhkyw/dataset/8
Runtime -> T4 GPU. Upload `nutsbolts_dataset_320.zip` to /content first.

In [ ]:
!pip -q install ultralytics
!nvidia-smi -L
!unzip -q -o /content/nutsbolts_dataset_320.zip -d /content/
print(open('/content/ds3/data.yaml').read())

In [ ]:
from ultralytics import YOLO
m = YOLO('yolo11n.pt')
m.train(data='/content/ds3/data.yaml', imgsz=320, epochs=60, batch=32, patience=15, seed=0,
        hsv_h=0.05, hsv_s=0.8, hsv_v=0.5, degrees=180, flipud=0.5, fliplr=0.5, scale=0.6, translate=0.15,
        mosaic=1.0, mixup=0.1, close_mosaic=10, erasing=0.2, project='/content/runs', name='nb_short')

In [ ]:
best = YOLO('/content/runs/nb_short/weights/best.pt')
val = best.val(data='/content/ds3/data.yaml', imgsz=320, split='val')
test = best.val(data='/content/ds3/data.yaml', imgsz=320, split='test')
print('VAL  mAP50 %.3f P %.3f R %.3f' % (val.box.map50, val.box.mp, val.box.mr))
print('TEST mAP50 %.3f P %.3f R %.3f' % (test.box.map50, test.box.mp, test.box.mr))
best.export(format='onnx', imgsz=320)

In [ ]:
!cd /content/runs/nb_short && zip -qr /content/nb_short_results.zip weights/best.pt weights/best.onnx results.csv confusion_matrix.png confusion_matrix_normalized.png results.png args.yaml
from google.colab import files
files.download('/content/nb_short_results.zip')